# Test RAG Agents

This notebook tests the RAG (Retrieval Augmented Generation) implementation in agent nodes.

In [ ]:
from sahiloan_chatbot.application.chat_service.workflow.nodes import Nodes
from sahiloan_chatbot.application.chat_service.workflow.state import ChatState
from sahiloan_chatbot.domain.tools import PineconeRetriever

## 1. Test Retriever

In [ ]:
# Initialize retriever
retriever = PineconeRetriever(top_k=3)

# Test query
query = "What is the difference between Home Loan and LAP?"
chunks = retriever.retrieve(query)

print(f"Query: {query}\n")
print(f"Retrieved {len(chunks)} relevant chunks:\n")

for i, chunk in enumerate(chunks, 1):
    print(f"Chunk {i} (Score: {chunk['score']:.4f}, Source: {chunk['source']})")
    print(f"{chunk['text'][:200]}...\n")
    print("-"*80)

## 2. Test Complete Agent Flow

In [ ]:
# Initialize nodes
nodes = Nodes()

# Test queries for different agents
test_cases = [
    {
        "query": "What is the difference between Home Loan and LAP?",
        "agent": "loan_agent",
        "description": "Loan Agent - Home Loan vs LAP"
    },
    {
        "query": "How is Sahiloan different from banks?",
        "agent": "general_agent", 
        "description": "General Agent - About Sahiloan"
    },
    {
        "query": "What documents do I need for a loan?",
        "agent": "document_agent",
        "description": "Document Agent - Required docs"
    }
]

for test in test_cases:
    print("="*80)
    print(f"TEST: {test['description']}")
    print("="*80)
    print(f"\nQuery: {test['query']}\n")
    
    # Create state with user message
    state = ChatState(
        messages=[{"role": "user", "content": test['query']}]
    )
    
    # Call appropriate agent
    if test['agent'] == "loan_agent":
        result_state = nodes.loan_agent(state)
    elif test['agent'] == "general_agent":
        result_state = nodes.general_agent(state)
    elif test['agent'] == "document_agent":
        result_state = nodes.document_agent(state)
    
    # Display response
    response = result_state["messages"][-1]["content"]
    print(f"Response:\n{response}\n")
    print("="*80 + "\n")

In [ ]:
# Test with intent routing
user_query = "How can I reduce my EMI?"

print(f"User Query: {user_query}\n")

# Step 1: Intent routing
state = ChatState(messages=[{"role": "user", "content": user_query}])
state = nodes.intent_router(state)

print(f"Routed to: {state['route_to']}\n")

# Step 2: Get response from appropriate agent
if state['route_to'] == 'loan_agent':
    state = nodes.loan_agent(state)
elif state['route_to'] == 'general_agent':
    state = nodes.general_agent(state)
elif state['route_to'] == 'document_agent':
    state = nodes.document_agent(state)

# Display final response
print(f"Agent Response:\n{state['messages'][-1]['content']}")